#### Feature Matrix v2 — Hourly Contemporaneous Model




In [1]:
import pandas as pd
import numpy as np

PARQUET   = '../data/all_thermal_with_weather.parquet'
LOAD_PATH = '../data/load_germany_hourly.parquet'
OUT_PATH  = '../data/features_v2.parquet'

WEATHER_VARS = [
    'temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall',
    'wind_speed_10m', 'wind_gusts_10m', 'surface_pressure',
    'shortwave_radiation', 'cloud_cover'
]

#### Load and Sort

In [2]:
print('Loading data...')
df = pd.read_parquet(PARQUET)
df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values(['EICCode', 'DateTime']).reset_index(drop=True)
print(f'  Loaded: {df.shape}')
print(f'  Columns: {df.columns.tolist()}')

Loading data...
  Loaded: (12589440, 29)
  Columns: ['DateTime', 'Fuel', 'EICCode', 'Unit', 'UnitID', 'ActualGenerationOutput', 'AvailableCapacity_eex_ex_post', 'AvailableCapacity_eex_real_time', 'AvailableCapacity_entsoe_ex_post', 'AvailableCapacity_entsoe_real_time', 'Event_eex_ex_post', 'Event_eex_real_time', 'Event_entsoe_ex_post', 'Event_entsoe_real_time', 'InstalledCapacity', 'UnavailableCapacity_eex_ex_post', 'UnavailableCapacity_eex_real_time', 'Type_entsoe_ex_post', 'latitude', 'longitude', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'wind_gusts_10m', 'surface_pressure', 'shortwave_radiation', 'cloud_cover']


#### Onset Detection

`is_onset = 1` for the first hour of each unplanned event.  
Continuation hours (hours 2+ of an active unplanned event) are **dropped** so the model only sees hours where a unit is at risk of a new onset.

In [3]:
df['is_unplanned']   = (df['Type_entsoe_ex_post'] == 'Unplanned').astype(int)
df['prev_unplanned'] = df.groupby('EICCode')['is_unplanned'].shift(1).fillna(0).astype(int)
df['prev_time']      = df.groupby('EICCode')['DateTime'].shift(1)
df['gap_hours']      = (df['DateTime'] - df['prev_time']).dt.total_seconds() / 3600

# Onset = first hour of unplanned event
df['is_onset'] = (
    (df['is_unplanned'] == 1) &
    ((df['prev_unplanned'] == 0) | (df['gap_hours'] > 1))
).astype(int)

# Continuation = unplanned but NOT onset
df['is_continuation'] = ((df['is_unplanned'] == 1) & (df['is_onset'] == 0)).astype(int)

n_total        = len(df)
n_onset        = df['is_onset'].sum()
n_continuation = df['is_continuation'].sum()
n_working      = n_total - n_continuation

print(f'Total rows before drop  : {n_total:,}')
print(f'Onset hours             : {n_onset:,}')
print(f'Continuation hours drop : {n_continuation:,}')
print(f'Working dataset rows    : {n_working:,}')
print(f'Onset rate              : {n_onset / n_working * 100:.4f}%')
print(f'Imbalance ratio         : {(n_working - n_onset) / n_onset:.0f} : 1')

working = df[df['is_continuation'] == 0].copy()
working = working.drop(columns=['is_unplanned', 'is_continuation', 'prev_unplanned', 'prev_time', 'gap_hours'])
working = working.sort_values(['EICCode', 'DateTime']).reset_index(drop=True)
print(f'\nWorking dataset: {working.shape}')

Total rows before drop  : 12,589,440
Onset hours             : 9,832
Continuation hours drop : 654,041
Working dataset rows    : 11,935,399
Onset rate              : 0.0824%
Imbalance ratio         : 1213 : 1

Working dataset: (11935399, 30)


#### Local Temperature Percentile Bins



In [5]:
TRAIN_YEAR_MAX = 2021

train_mask = working['DateTime'].dt.year <= TRAIN_YEAR_MAX
train_df   = working[train_mask]

# Build quantile map per unit on training data only
quantile_map = {}
for unit, grp in train_df.groupby('EICCode'):
    temps = grp['temperature_2m'].dropna()
    if len(temps) < 10:
        quantile_map[unit] = None
        continue
    quantile_map[unit] = {
        'p2':  temps.quantile(0.02),
        'p5':  temps.quantile(0.05),
        'p10': temps.quantile(0.10),
        'p90': temps.quantile(0.90),
        'p95': temps.quantile(0.95),
        'p98': temps.quantile(0.98),
    }

print(f'Quantile map built for {sum(v is not None for v in quantile_map.values())} / {len(quantile_map)} units')

# Assign bins vectorized per unit
working['local_temp_pct_bin'] = np.int8(3)  # default = reference bin

for unit, qmap in quantile_map.items():
    if qmap is None:
        continue
    unit_mask = working['EICCode'] == unit
    temps = working.loc[unit_mask, 'temperature_2m']

    # Build bin edges: [-inf, p2, p5, p10, p90, p95, p98, +inf]
    bin_edges = [
        -np.inf,
        qmap['p2'], qmap['p5'], qmap['p10'],
        qmap['p90'], qmap['p95'], qmap['p98'],
        np.inf
    ]
    labels = [0, 1, 2, 3, 4, 5, 6]
    binned = pd.cut(temps, bins=bin_edges, labels=labels, include_lowest=True)
    working.loc[unit_mask, 'local_temp_pct_bin'] = binned.astype(np.int8)

working['local_temp_pct_bin'] = working['local_temp_pct_bin'].astype(int)
print(f'\nlocal_temp_pct_bin distribution:')
print(working['local_temp_pct_bin'].value_counts().sort_index())

# VALIDATION: onset rate per bin — bin 6 must be highest
print('\n=== VALIDATION: Onset rate per local_temp_pct_bin ===')
for b in range(7):
    mask = working['local_temp_pct_bin'] == b
    n    = mask.sum()
    rate = working.loc[mask, 'is_onset'].mean() * 100
    print(f'  bin {b}: n={n:>10,}, onset_rate={rate:.4f}%')

bin6_rate = working.loc[working['local_temp_pct_bin'] == 6, 'is_onset'].mean()
all_rates = [working.loc[working['local_temp_pct_bin'] == b, 'is_onset'].mean() for b in range(7)]
if bin6_rate != max(all_rates):
    print('\n*** WARNING: Bin 6 does NOT have the highest onset rate — check feature logic! ***')
else:
    print('\n[OK] Bin 6 has the highest onset rate.')

Quantile map built for 179 / 179 units

local_temp_pct_bin distribution:
local_temp_pct_bin
0     227756
1     355115
2     568696
3    9571633
4     605453
5     364438
6     242308
Name: count, dtype: int64

=== VALIDATION: Onset rate per local_temp_pct_bin ===
  bin 0: n=   227,756, onset_rate=0.1326%
  bin 1: n=   355,115, onset_rate=0.0893%
  bin 2: n=   568,696, onset_rate=0.0874%
  bin 3: n= 9,571,633, onset_rate=0.0779%
  bin 4: n=   605,453, onset_rate=0.0854%
  bin 5: n=   364,438, onset_rate=0.0988%
  bin 6: n=   242,308, onset_rate=0.1593%

[OK] Bin 6 has the highest onset rate.


#### Weather Features

Raw weather (9) + rolling temperature (3) + compound weather (2 of 3 — `fuel_gas_x_temp` computed after fuel dummies in Step 7)

In [6]:
# Raw weather: already present as-is
print('Raw weather columns present:', [c for c in WEATHER_VARS if c in working.columns])

# Rolling temperature per unit — past-only (includes current row, which is contemporaneous)
print('Computing rolling temperature features (may take ~1 min)...')
working = working.sort_values(['EICCode', 'DateTime'])

working['rolling_temp_max_6h'] = (
    working.groupby('EICCode')['temperature_2m']
    .transform(lambda x: x.rolling(6, min_periods=1).max())
)
working['rolling_temp_max_24h'] = (
    working.groupby('EICCode')['temperature_2m']
    .transform(lambda x: x.rolling(24, min_periods=1).max())
)
working['rolling_temp_max_48h'] = (
    working.groupby('EICCode')['temperature_2m']
    .transform(lambda x: x.rolling(48, min_periods=1).max())
)
print('Rolling temperature features added: rolling_temp_max_6h, rolling_temp_max_24h, rolling_temp_max_48h')

# Compound weather (2 of 3 — fuel_gas_x_temp computed after Step 7)
working['temp_x_humidity']   = working['temperature_2m'] * working['relative_humidity_2m']
working['cold_stress_index'] = np.maximum(0, 5 - working['temperature_2m']) * working['wind_speed_10m']
print('Compound features added: temp_x_humidity, cold_stress_index')
print('  (fuel_gas_x_temp will be added after Step 7)')

Raw weather columns present: ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'wind_gusts_10m', 'surface_pressure', 'shortwave_radiation', 'cloud_cover']
Computing rolling temperature features (may take ~1 min)...
Rolling temperature features added: rolling_temp_max_6h, rolling_temp_max_24h, rolling_temp_max_48h
Compound features added: temp_x_humidity, cold_stress_index
  (fuel_gas_x_temp will be added after Step 7)


#### Capacity Features

Fixed plant characteristics only — no operational metrics (utilisation_rate etc.) to keep the model contemporaneous.

In [7]:
# InstalledCapacity: as-is (fixed plant characteristic)
print(f'InstalledCapacity: {working["InstalledCapacity"].describe().round(1).to_string()}')

# capacity_size: 0=small (<200MW), 1=medium (200-600MW), 2=large (>600MW)
working['capacity_size'] = pd.cut(
    working['InstalledCapacity'],
    bins=[0, 200, 600, 9999],
    labels=[0, 1, 2]
).astype(int)

print(f'\ncapacity_size distribution:')
print(working['capacity_size'].value_counts().sort_index())

InstalledCapacity: count    11935399.0
mean          411.3
std           258.3
min            10.2
25%           167.0
50%           383.0
75%           605.0
max          1060.0

capacity_size distribution:
capacity_size
0    3734932
1    5109401
2    3091066
Name: count, dtype: int64


Operational History

`n_onsets_last_90d`: count of unplanned onsets in the 90 days **strictly before** the current timestamp, per unit.  
Uses vectorized searchsorted — loops over ~83 units, not over 12M rows.

In [8]:
print('Computing n_onsets_last_90d (vectorized searchsorted)...')
working = working.sort_values(['EICCode', 'DateTime']).reset_index(drop=True)

# Collect onset timestamps per unit
onset_df = (
    working[working['is_onset'] == 1][['EICCode', 'DateTime']]
    .sort_values(['EICCode', 'DateTime'])
)

n_90d      = np.zeros(len(working), dtype=np.int32)
cutoff_ns  = np.int64(90 * 24 * 3600 * int(1e9))  # 90 days in nanoseconds

for unit, grp in working.groupby('EICCode'):
    unit_onset_times = (
        onset_df[onset_df['EICCode'] == unit]['DateTime']
        .values.astype(np.int64)
    )
    if len(unit_onset_times) == 0:
        continue

    times = grp['DateTime'].values.astype(np.int64)
    idx   = grp.index.values

    # onsets strictly before t
    right = np.searchsorted(unit_onset_times, times, side='left')
    # onsets >= t - 90d
    left  = np.searchsorted(unit_onset_times, times - cutoff_ns, side='left')

    n_90d[idx] = right - left

working['n_onsets_last_90d'] = n_90d
print('n_onsets_last_90d added')
print(working['n_onsets_last_90d'].describe().round(2).to_string())

Computing n_onsets_last_90d (vectorized searchsorted)...
n_onsets_last_90d added
count    11935399.00
mean            1.61
std             2.64
min             0.00
25%             0.00
50%             0.00
75%             2.00
max            41.00


#### Step 7 — Fuel Dummies

Reference category = **Fossil Hard coal** (dropped).  
Dummies retained: `fuel_Fossil Gas`, `fuel_Fossil Brown coal/Lignite`

In [9]:
# Get all three dummies then drop Hard coal (Hard coal = reference category)
fuel_dummies = pd.get_dummies(working['Fuel'], prefix='fuel')
print('All fuel dummy columns:', fuel_dummies.columns.tolist())

# Drop Hard coal to make it the reference
fuel_dummies = fuel_dummies.drop(columns=['fuel_Fossil Hard coal'])
print('Kept dummies:', fuel_dummies.columns.tolist())

working = pd.concat([working, fuel_dummies], axis=1)
print(fuel_dummies.sum().to_string())

# Compound weather: fuel_gas_x_temp (requires fuel dummy)
working['fuel_gas_x_temp'] = working['fuel_Fossil Gas'] * working['temperature_2m']
print('\nfuel_gas_x_temp added')
print(working['fuel_gas_x_temp'].describe().round(2).to_string())

All fuel dummy columns: ['fuel_Fossil Brown coal/Lignite', 'fuel_Fossil Gas', 'fuel_Fossil Hard coal']
Kept dummies: ['fuel_Fossil Brown coal/Lignite', 'fuel_Fossil Gas']
fuel_Fossil Brown coal/Lignite    2956763
fuel_Fossil Gas                   4970311

fuel_gas_x_temp added
count    11935399.00
mean            4.58
std             7.36
min           -20.80
25%             0.00
50%             0.00
75%             8.40
max            39.90


#### Step 8 — Load Features

`rolling_load_24h` computed on the load dataframe before joining (no gaps in national hourly load data).

In [10]:
load = pd.read_parquet(LOAD_PATH)
load['DateTime'] = pd.to_datetime(load['DateTime'])
load = load.sort_values('DateTime').reset_index(drop=True)

# Compute rolling_load_24h on load df before joining (no gaps in national data)
load['rolling_load_24h'] = load['load_mwh'].rolling(24, min_periods=1).mean()

print(f'Load data shape: {load.shape}')
print(load[['load_mwh', 'rolling_load_24h']].describe().round(1).to_string())

working = working.merge(load[['DateTime', 'load_mwh', 'rolling_load_24h']], on='DateTime', how='left')
print(f'\nAfter join: {working.shape}')
print(f'Missing load_mwh      : {working["load_mwh"].isna().sum():,}')
print(f'Missing rolling_load_24h: {working["rolling_load_24h"].isna().sum():,}')

Load data shape: (87662, 3)
       load_mwh  rolling_load_24h
count   87662.0           87662.0
mean    56026.0           56024.8
std      9987.3            6618.4
min     30902.8           37429.9
25%     47828.8           51420.2
50%     55742.6           56643.2
75%     64242.8           60711.1
max     81319.5           72511.4

After join: (11935399, 43)
Missing load_mwh      : 1,366
Missing rolling_load_24h: 1,366


## Step 9 — Structural Control

In [10]:
working['year'] = working['DateTime'].dt.year
print('year added')
print(working['year'].value_counts().sort_index().to_string())

year added
year
2015    1343909
2016    1294005
2017    1305138
2018    1274602
2019    1217469
2020    1129280
2021    1133924
2022    1127257
2023    1078022
2024    1031793


## Step 10 — Save

In [11]:
FEATURE_COLS = [
    # Weather raw (9)
    'temperature_2m', 'relative_humidity_2m', 'precipitation',
    'snowfall', 'wind_speed_10m', 'wind_gusts_10m',
    'surface_pressure', 'shortwave_radiation', 'cloud_cover',
    # Weather engineered (4)
    'local_temp_pct_bin',
    'rolling_temp_max_6h',
    'rolling_temp_max_24h',
    'rolling_temp_max_48h',
    # Compound weather (3)
    'temp_x_humidity',
    'cold_stress_index',
    'fuel_gas_x_temp',
    # Capacity (2)
    'InstalledCapacity',
    'capacity_size',
    # Operational history (1)
    'n_onsets_last_90d',
    # Load (2)
    'load_mwh',
    'rolling_load_24h',
    # Structural (1)
    'year',
    # Fuel dummies (2)
    'fuel_Fossil Gas',
    'fuel_Fossil Brown coal/Lignite',
]

ID_COLS = ['EICCode', 'DateTime', 'Fuel', 'Type_entsoe_ex_post']

assert len(FEATURE_COLS) == 24, f'Expected 24 features, got {len(FEATURE_COLS)}'

# Verify all columns exist
missing_cols = [c for c in FEATURE_COLS if c not in working.columns]
if missing_cols:
    raise ValueError(f'Missing columns: {missing_cols}')

output = working[ID_COLS + FEATURE_COLS + ['is_onset']].copy()
output.to_parquet(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {output.shape}')

Saved: ../data/features_v2.parquet
Shape: (11935399, 29)


## Step 11 — Validation

In [14]:




# 1. Shape
print(f'\n1. Feature matrix shape : {output.shape}')

# 2. Feature count
print(f'2. Number of features   : {len(FEATURE_COLS)}  (must be 24)')

# 3. Onset rate
onset_rate = output['is_onset'].mean() * 100
print(f'3. Onset rate           : {onset_rate:.4f}%')

# 4. Imbalance ratio
n_onset     = output['is_onset'].sum()
n_non_onset = (output['is_onset'] == 0).sum()
print(f'4. Imbalance ratio      : {n_non_onset / n_onset:.0f} : 1')

# 5. Missing values
print('\n5. Missing values per feature:')
missing = output[FEATURE_COLS].isnull().sum()
if missing.any():
    print(missing[missing > 0].to_string())
else:
    print('   None')

# 6. local_temp_pct_bin: count and onset rate per bin
print('\n6. local_temp_pct_bin distribution:')
print(f'   {"Bin":>4}  {"Count":>12}  {"Onset rate":>12}')
for b in range(7):
    mask  = output['local_temp_pct_bin'] == b
    n     = mask.sum()
    rate  = output.loc[mask, 'is_onset'].mean() * 100
    print(f'   {b:>4}  {n:>12,}  {rate:>11.4f}%')

all_rates  = [output.loc[output['local_temp_pct_bin'] == b, 'is_onset'].mean() for b in range(7)]
bin6_rate  = all_rates[6]
if bin6_rate == max(all_rates):
    print('   [OK] Bin 6 has the highest onset rate.')
else:
    max_bin = int(np.argmax(all_rates))
    print(f'   *** WARNING: Bin {max_bin} has the highest onset rate ({all_rates[max_bin]*100:.4f}%), not bin 6 ({bin6_rate*100:.4f}%) ***')

# 7. n_onsets_last_90d
print('\n7. n_onsets_last_90d distribution:')
print(output['n_onsets_last_90d'].describe().round(2).to_string())

# 8. year distribution
print('\n8. year distribution:')
print(output['year'].value_counts().sort_index().to_string())




1. Feature matrix shape : (11935399, 29)
2. Number of features   : 24  (must be 24)
3. Onset rate           : 0.0824%
4. Imbalance ratio      : 1213 : 1

5. Missing values per feature:
load_mwh            1366
rolling_load_24h    1366

6. local_temp_pct_bin distribution:
    Bin         Count    Onset rate
      0       227,756       0.1326%
      1       355,115       0.0893%
      2       568,696       0.0874%
      3     9,571,633       0.0779%
      4       605,453       0.0854%
      5       364,438       0.0988%
      6       242,308       0.1593%
   [OK] Bin 6 has the highest onset rate.

7. n_onsets_last_90d distribution:
count    11935399.00
mean            1.61
std             2.64
min             0.00
25%             0.00
50%             0.00
75%             2.00
max            41.00

8. year distribution:
year
2015    1343909
2016    1294005
2017    1305138
2018    1274602
2019    1217469
2020    1129280
2021    1133924
2022    1127257
2023    1078022
2024    1031793
